In [ ]:
from __future__ import annotations

import cirq
import numpy as np
from cirq.ops import GlobalPhaseGate

# Needed for text/SVG diagrams: top-level ``cirq.GlobalPhaseGate`` can be missing in some kernels.
setattr(cirq, "GlobalPhaseGate", GlobalPhaseGate)

import sympy
from IPython.display import display

from cirq.contrib.svg import SVGCircuit

# Shared sampling/CDR settings used across later notebook cells.
GLOBAL_NUM_SHOTS = 8192
CDR_NUM_TRAINING_CIRCUITS = 50
CDR_T_MAX_GRADIENT = 21
CDR_T_MAX_VQE = CDR_T_MAX_GRADIENT
CDR_NON_CLIFFORD_PARAMS_MAX = CDR_T_MAX_VQE
CDR_TARGET_PHASE_NONCLIFF_SNAP_FRACTION = 0
CDR_BASE_SEED = 42
GLOBAL_RANDOM_SEED = 1234
GLOBAL_SAMPLING_SEED = 1234
GLOBAL_MEASUREMENT_SCHEME = "ogm"
GLOBAL_APPLY_READOUT_NOISE = True
GLOBAL_EPSILON = 0.1
SHADOWGROUPING_ROOT = "/Users/zacharyhe/shadowgrouping"
GLOBAL_READOUT_P0_SUCCESS = np.array([0.97, 0.96, 0.93, 0.96, 0.92, 0.93, 0.94, 0.92])
GLOBAL_READOUT_P1_SUCCESS = np.array([0.85, 0.90, 0.88, 0.90, 0.86, 0.89, 0.87, 0.85])


In [ ]:
# construct the circuit

def prepare_ansatz_cirq(num_spatial_orbitals, num_layers=1):
    # Map Alpha to Row 0, Beta to Row 1
    qubits = [cirq.GridQubit(0, i) for i in range(num_spatial_orbitals)] + \
             [cirq.GridQubit(1, i) for i in range(num_spatial_orbitals)]

    circuit = cirq.Circuit()
    p_idx = 0

    # --- STEP 1: INITIAL STATE ---
    initial_gates = [cirq.X(qubits[i]) for i in range(1, len(qubits), 2)]
    circuit.append(initial_gates)

    for layer in range(num_layers):
        # --- STEP 2: GIVENS ROTATION G (Even-Odd) ---
        even_odd_moments = []
        for i in range(0, num_spatial_orbitals - 1, 2):
            theta = sympy.Symbol(f"th_{layer}_{p_idx}")
            even_odd_moments.append(cirq.FSimGate(theta, 0).on(qubits[i], qubits[i + 1]))
            even_odd_moments.append(
                cirq.FSimGate(theta, 0).on(
                    qubits[i + num_spatial_orbitals], qubits[i + 1 + num_spatial_orbitals]
                )
            )
            p_idx += 1

        circuit.append(even_odd_moments, strategy=cirq.InsertStrategy.NEW_THEN_INLINE)

        # --- STEP 3: GIVENS ROTATION G' (Odd-Even) ---
        odd_even_moments = []
        for i in range(1, num_spatial_orbitals - 1, 2):
            theta = sympy.Symbol(f"th_{layer}_{p_idx}")
            odd_even_moments.append(cirq.FSimGate(theta, 0).on(qubits[i], qubits[i + 1]))
            odd_even_moments.append(
                cirq.FSimGate(theta, 0).on(
                    qubits[i + num_spatial_orbitals], qubits[i + 1 + num_spatial_orbitals]
                )
            )
            p_idx += 1

        circuit.append(odd_even_moments, strategy=cirq.InsertStrategy.NEW_THEN_INLINE)

        # --- STEP 4: ON-SITE POTENTIAL (Double Occupancy) ---
        onsite_moments = []
        for i in range(num_spatial_orbitals):
            phi = sympy.Symbol(f"ph_{layer}_{p_idx}")
            q_alpha = qubits[i]
            q_beta = qubits[i + num_spatial_orbitals]
            onsite_moments.append(cirq.FSimGate(0, phi).on(q_alpha, q_beta))
            p_idx += 1

        circuit.append(onsite_moments, strategy=cirq.InsertStrategy.NEW_THEN_INLINE)

    return circuit, qubits

# --- Test Case ---
num_spatial = 4
num_layers = 1
ansatz_circuit, ansatz_qubits = prepare_ansatz_cirq(
    num_spatial_orbitals=num_spatial, num_layers=num_layers
)

# Diagram with a random angle for each symbolic parameter.
rng_viz = np.random.default_rng(GLOBAL_RANDOM_SEED)
param_map = {
    name: float(rng_viz.uniform(0, 2 * np.pi))
    for name in sorted(cirq.parameter_names(ansatz_circuit))
}

resolver_viz = cirq.ParamResolver(param_map)
circuit_draw = cirq.resolve_parameters(ansatz_circuit, resolver_viz)

print("--- Cirq FSim Ansatz (random viz params) ---")
print("param_map:", param_map)
print(circuit_draw)
print(f"\nCircuit Depth: {len(ansatz_circuit)} moments")
display(SVGCircuit(circuit_draw))

In [ ]:
import sys
from pathlib import Path

# Repo root resolution for local file loading.
_repo = Path.cwd().resolve()
if not (_repo / "main_cursor_lib.py").is_file():
    _repo = _repo.parent
sys.path.insert(0, str(_repo))

import main_cursor_lib as mainlib
from decompose_fsim_gate import (
    decompose_fsim_phi_only,
    decompose_fsim_theta_only,
    is_cz_plus_single_qubit,
)
from main_cursor_lib import (
    CZ_ONSITE_TAG,
    DEFAULT_HIGH_CZ_MULTIPLIER,
    GateArityDepolarizingNoise,
    ONE_QUBIT_GATE_DEPOL_PROB,
    TWO_QUBIT_GATE_DEPOL_PROB,
    cz_tag_for_horizontal_pair,
    ordered_parameter_symbols,
    prepare_decomposed_ansatz_cirq,
    prepare_original_fsim_ansatz_cirq,
    trace_energy,
)


def load_pauli_sum_from_numbered_file(path: Path, qubits: list[cirq.Qid]) -> cirq.PauliSum:
    """Read ``coeff q0 q1 ...`` lines (0=I, 1=X, 2=Y, 3=Z) into a ``PauliSum``."""
    idx_to_pauli = {1: cirq.X, 2: cirq.Y, 3: cirq.Z}
    out = cirq.PauliSum()

    with path.open("r", encoding="utf-8") as f:
        for lineno, raw in enumerate(f, start=1):
            line = raw.strip()
            if not line:
                continue

            parts = line.split()
            coeff = float(parts[0])
            pauli_codes = [int(x) for x in parts[1:]]

            if len(pauli_codes) != len(qubits):
                raise ValueError(
                    f"{path}:{lineno} has {len(pauli_codes)} Pauli indices, expected {len(qubits)}."
                )

            pauli_string = cirq.PauliString()
            for q, code in zip(qubits, pauli_codes):
                if code == 0:
                    continue
                if code not in idx_to_pauli:
                    raise ValueError(
                        f"{path}:{lineno} has invalid Pauli code {code}; expected 0/1/2/3."
                    )
                pauli_string *= idx_to_pauli[code](q)

            out += coeff * pauli_string

    return out


def ordered_parameter_symbols(num_spatial: int, num_layers: int) -> list[sympy.Symbol]:
    """Match ``prepare_ansatz_cirq`` symbol naming (cumulative ``p_idx`` across layers)."""
    symbols: list[sympy.Symbol] = []
    p_idx = 0
    for layer in range(num_layers):
        for _ in range(0, num_spatial - 1, 2):
            symbols.append(sympy.Symbol(f"th_{layer}_{p_idx}"))
            p_idx += 1
        for _ in range(1, num_spatial - 1, 2):
            symbols.append(sympy.Symbol(f"th_{layer}_{p_idx}"))
            p_idx += 1
        for _ in range(num_spatial):
            symbols.append(sympy.Symbol(f"ph_{layer}_{p_idx}"))
            p_idx += 1
    return symbols


def params_per_layer(num_spatial: int) -> int:
    return (num_spatial // 2) + ((num_spatial - 1) // 2) + num_spatial


def _tag_cz_ops(ops: list[cirq.Operation], tag: str) -> list[cirq.Operation]:
    tagged: list[cirq.Operation] = []
    for op in ops:
        if isinstance(op.gate, cirq.CZPowGate) and np.isclose(float(op.gate.exponent), 1.0):
            tagged.append(op.with_tags(tag))
        else:
            tagged.append(op)
    return tagged


def decompose_ansatz_fsim_ops(resolved: cirq.Circuit) -> cirq.Circuit:
    """Replace every resolved FSim(theta,0)/(0,phi) with CZ + 1Q gates and CZ tags."""
    decomposed = cirq.Circuit()
    for moment in resolved:
        moment_ops: list[cirq.Operation] = []
        for op in moment.operations:
            gate = op.gate
            if isinstance(gate, cirq.FSimGate):
                theta = float(gate.theta)
                phi = float(gate.phi)
                q0, q1 = op.qubits

                if np.isclose(phi, 0.0):
                    theta_ops = decompose_fsim_theta_only(theta, q0, q1)
                    theta_tag = cz_tag_for_horizontal_pair(q0, q1)
                    moment_ops.extend(_tag_cz_ops(theta_ops, theta_tag))
                elif np.isclose(theta, 0.0):
                    phi_ops = decompose_fsim_phi_only(phi, q0, q1)
                    moment_ops.extend(_tag_cz_ops(phi_ops, CZ_ONSITE_TAG))
                else:
                    raise ValueError(
                        "Encountered general FSim(theta, phi); expected theta-only or phi-only."
                    )
            else:
                moment_ops.append(op)

        decomposed.append(moment_ops)

    return decomposed


# --- H4 @ bond length 2 ---
H_atom = 4
bond_length = 2
num_spatial = 4
ansatz_layers = 3

ham_path = _repo / "Pauli_Ham" / f"H{H_atom}_bond_{bond_length}_number_convention.txt"

# Layer-3 baseline vector (same as old_H4/main_cursor.ipynb).
vqe_parameters = np.array([
    1.02328779,
    1.33825349,
    0.78981901,
    0.36255437,
    0.64652655,
    3.31926241,
    -0.07988963,
    0.22301099,
    0.4992053,
    1.3831351,
    5.50652141,
    2.7144864,
    2.14070825,
    -0.10898955,
    -0.2443687,
    -0.45224915,
    0.95313939,
    -1.18188991,
    -1.73092033,
    1.36303661,
    1.41789702,
])

ppl = params_per_layer(num_spatial)
expected_num_parameters = ansatz_layers * ppl
if len(vqe_parameters) != expected_num_parameters:
    raise ValueError(
        f"ansatz_layers={ansatz_layers} requires {expected_num_parameters} parameters "
        f"({ppl} per layer), got {len(vqe_parameters)}."
    )

# Build 3-layer FSim ansatz and helper-driven symbolic decomposition template.
energy_circuit, energy_qubits = prepare_original_fsim_ansatz_cirq(
    num_spatial_orbitals=num_spatial, num_layers=ansatz_layers
)
decomposed_template, decomp_qubits = prepare_decomposed_ansatz_cirq(
    num_spatial_orbitals=num_spatial, num_layers=ansatz_layers
)
if list(decomp_qubits) != list(energy_qubits):
    raise ValueError("Decomposed template qubits do not match original ansatz qubit order.")

symbols = mainlib.ordered_parameter_symbols(num_spatial, ansatz_layers)
resolver = cirq.ParamResolver(dict(zip(symbols, vqe_parameters)))
resolved_circuit = cirq.resolve_parameters(energy_circuit, resolver)
decomposed_circuit = cirq.resolve_parameters(decomposed_template, resolver)
if not is_cz_plus_single_qubit(decomposed_circuit.all_operations()):
    raise ValueError("Decomposed circuit contains non-(CZ + 1Q) gates.")

sim = cirq.Simulator()

psi = np.asarray(
    sim.simulate(resolved_circuit, qubit_order=energy_qubits).final_state_vector,
    dtype=np.complex128,
)
psi_decomp = np.asarray(
    sim.simulate(decomposed_circuit, qubit_order=energy_qubits).final_state_vector,
    dtype=np.complex128,
)

pauli_sum = load_pauli_sum_from_numbered_file(ham_path, list(energy_qubits))
qubit_map = {q: i for i, q in enumerate(energy_qubits)}
e_vqe = float(np.real(pauli_sum.expectation_from_state_vector(psi, qubit_map=qubit_map)))
e_vqe_decomp = float(
    np.real(pauli_sum.expectation_from_state_vector(psi_decomp, qubit_map=qubit_map))
)
e_gs = float(np.linalg.eigvalsh(pauli_sum.matrix(qubits=energy_qubits))[0].real)

print(f"\nH{H_atom} bond {bond_length} (Hartree-Fock geometry)")
print(f"Hamiltonian source: {ham_path}")
print(f"Ansatz: {ansatz_layers} layers, {len(vqe_parameters)} parameters")
print(f"⟨H⟩ undecomposed (noiseless): {e_vqe:.10f} Eh")
print(f"⟨H⟩ decomposed CZ+1Q (noiseless): {e_vqe_decomp:.10f} Eh")
print(f"Energy diff (decomp - undecomp): {e_vqe_decomp - e_vqe:.3e} Eh")
print(f"Ground-state energy e_gs (exact): {e_gs:.10f} Eh")
print(f"ΔE undecomp = ⟨H⟩ - e_gs: {e_vqe - e_gs:.10f} Eh")
print(f"ΔE decomp   = ⟨H⟩ - e_gs: {e_vqe_decomp - e_gs:.10f} Eh")

In [ ]:
# Draw decomposed circuit (text + SVG)
print("--- Decomposed CZ + 1Q circuit ---")
print(decomposed_circuit)
print(f"\nCircuit Depth: {len(decomposed_circuit)} moments")
display(SVGCircuit(decomposed_circuit))

In [ ]:
def merge_adjacent_local_commuting(circuit: cirq.Circuit, atol: float = 1e-12) -> tuple[cirq.Circuit, dict[str, int]]:
    """Peephole simplifier on adjacent moments.

    Rules:
    1) Merge adjacent Rx/Ry/Rz on same qubit.
    2) Cancel adjacent H-H on same qubit.
    3) Cancel adjacent CZ-CZ on same qubit pair.
    """
    rx_type = type(cirq.rx(0.0))
    ry_type = type(cirq.ry(0.0))
    rz_type = type(cirq.rz(0.0))

    moments = [cirq.Moment(m.operations) for m in circuit]
    stats = {
        "merge_rx": 0,
        "merge_ry": 0,
        "merge_rz": 0,
        "cancel_hh": 0,
        "cancel_czcz": 0,
    }

    def _remove_and_replace(m1: cirq.Moment, m2: cirq.Moment, op1: cirq.Operation, op2: cirq.Operation, new_op: cirq.Operation | None) -> tuple[cirq.Moment, cirq.Moment]:
        ops1 = [op for op in m1.operations if op != op1]
        ops2 = [op for op in m2.operations if op != op2]
        if new_op is not None:
            ops1.append(new_op)
        return cirq.Moment(ops1), cirq.Moment(ops2)

    while True:
        changed = False
        for i in range(len(moments) - 1):
            m1 = moments[i]
            m2 = moments[i + 1]

            # 1) CZ-CZ cancellation on same pair.
            for op1 in m1.operations:
                if len(op1.qubits) != 2 or not isinstance(op1.gate, cirq.CZPowGate):
                    continue
                if not np.isclose(float(op1.gate.exponent), 1.0, atol=atol):
                    continue

                q0, q1 = op1.qubits
                op2_q0 = m2.operation_at(q0)
                if op2_q0 is None or len(op2_q0.qubits) != 2:
                    continue
                if set(op2_q0.qubits) != {q0, q1}:
                    continue
                if not isinstance(op2_q0.gate, cirq.CZPowGate):
                    continue
                if not np.isclose(float(op2_q0.gate.exponent), 1.0, atol=atol):
                    continue

                moments[i], moments[i + 1] = _remove_and_replace(m1, m2, op1, op2_q0, None)
                stats["cancel_czcz"] += 1
                changed = True
                break

            if changed:
                break

            # 2) Single-qubit local rules on shared qubits.
            shared_qubits = sorted(set(m1.qubits) & set(m2.qubits), key=str)
            for q in shared_qubits:
                op1 = m1.operation_at(q)
                op2 = m2.operation_at(q)

                if op1 is None or op2 is None:
                    continue
                if len(op1.qubits) != 1 or len(op2.qubits) != 1:
                    continue

                # H-H cancellation.
                if op1.gate == cirq.H and op2.gate == cirq.H:
                    moments[i], moments[i + 1] = _remove_and_replace(m1, m2, op1, op2, None)
                    stats["cancel_hh"] += 1
                    changed = True
                    break

                # Merge same-axis rotations.
                if isinstance(op1.gate, rx_type) and isinstance(op2.gate, rx_type):
                    merged_angle = float(op1.gate.exponent + op2.gate.exponent) * np.pi
                    new_op = None if np.isclose(merged_angle, 0.0, atol=atol) else cirq.rx(merged_angle).on(q)
                    moments[i], moments[i + 1] = _remove_and_replace(m1, m2, op1, op2, new_op)
                    stats["merge_rx"] += 1
                    changed = True
                    break

                if isinstance(op1.gate, ry_type) and isinstance(op2.gate, ry_type):
                    merged_angle = float(op1.gate.exponent + op2.gate.exponent) * np.pi
                    new_op = None if np.isclose(merged_angle, 0.0, atol=atol) else cirq.ry(merged_angle).on(q)
                    moments[i], moments[i + 1] = _remove_and_replace(m1, m2, op1, op2, new_op)
                    stats["merge_ry"] += 1
                    changed = True
                    break

                if isinstance(op1.gate, rz_type) and isinstance(op2.gate, rz_type):
                    merged_angle = float(op1.gate.exponent + op2.gate.exponent) * np.pi
                    new_op = None if np.isclose(merged_angle, 0.0, atol=atol) else cirq.rz(merged_angle).on(q)
                    moments[i], moments[i + 1] = _remove_and_replace(m1, m2, op1, op2, new_op)
                    stats["merge_rz"] += 1
                    changed = True
                    break

            if changed:
                break

        if not changed:
            break

    merged_circuit = cirq.Circuit(m for m in moments if len(m.operations) > 0)
    return merged_circuit, stats


def count_1q_2q(circuit: cirq.Circuit) -> tuple[int, int]:
    one_q = sum(1 for op in circuit.all_operations() if len(op.qubits) == 1)
    two_q = sum(1 for op in circuit.all_operations() if len(op.qubits) == 2)
    return one_q, two_q


adj_merged_circuit, merge_stats = merge_adjacent_local_commuting(decomposed_circuit)

before_1q, before_2q = count_1q_2q(decomposed_circuit)
after_1q, after_2q = count_1q_2q(adj_merged_circuit)

sim_adj = cirq.Simulator()
psi_adj = np.asarray(
    sim_adj.simulate(adj_merged_circuit, qubit_order=energy_qubits).final_state_vector,
    dtype=np.complex128,
)

e_adj = float(np.real(pauli_sum.expectation_from_state_vector(psi_adj, qubit_map=qubit_map)))
fid_adj = float(np.abs(np.vdot(psi_decomp, psi_adj)) ** 2)

print("--- Adjacent local-commuting simplification ---")
print(f"Rx merges: {merge_stats['merge_rx']}")
print(f"Ry merges: {merge_stats['merge_ry']}")
print(f"Rz merges: {merge_stats['merge_rz']}")
print(f"H-H cancels: {merge_stats['cancel_hh']}")
print(f"CZ-CZ cancels: {merge_stats['cancel_czcz']}")
print(f"1Q gates: {before_1q} -> {after_1q} (delta {after_1q - before_1q})")
print(f"2Q gates: {before_2q} -> {after_2q} (delta {after_2q - before_2q})")
print(f"Energy diff (adj-merged - decomposed): {e_adj - e_vqe_decomp:.3e} Eh")
print(f"State fidelity |<psi_decomp|psi_adj>|^2: {fid_adj:.12f}")

print("\n--- Adjacent-merged circuit ---")
print(adj_merged_circuit)
print(f"\nCircuit Depth: {len(adj_merged_circuit)} moments")
display(SVGCircuit(adj_merged_circuit))

In [ ]:
# Noisy energy on the adjacent-merged decomposed circuit (gate depolarizing only).
hamiltonian_matrix = pauli_sum.matrix(qubits=energy_qubits)

high_cz_multiplier = DEFAULT_HIGH_CZ_MULTIPLIER
gate_noise = GateArityDepolarizingNoise(
    two_qubit_depol_prob=TWO_QUBIT_GATE_DEPOL_PROB,
    one_qubit_depol_prob=ONE_QUBIT_GATE_DEPOL_PROB,
    high_cz_multiplier=high_cz_multiplier,
)

num_tagged_high_cz = sum(
    1
    for op in adj_merged_circuit.all_operations()
    if isinstance(op.gate, cirq.CZPowGate)
    and np.isclose(float(op.gate.exponent), 1.0)
    and ("cz_high" in getattr(op, "tags", ()))
)

noisy_adj_merged = adj_merged_circuit.with_noise(gate_noise)
rho_noisy = np.asarray(
    cirq.DensityMatrixSimulator(seed=GLOBAL_RANDOM_SEED)
    .simulate(noisy_adj_merged, qubit_order=energy_qubits)
    .final_density_matrix,
    dtype=np.complex128,
)

e_noisy_adj = trace_energy(hamiltonian_matrix, rho_noisy)

print("--- Noisy adj-merged evaluation (gate depolarizing) ---")
print(
    f"noise: two_qubit_depol={gate_noise.two_qubit_depol_prob} "
    f"one_qubit_depol={gate_noise.one_qubit_depol_prob} "
    f"high_cz_multiplier={gate_noise.high_cz_multiplier}"
)
print(f"tagged high-CZ count in adj_merged_circuit: {num_tagged_high_cz}")
print(f"Tr[H rho_noisy(adj-merged)] = {e_noisy_adj:.10f} Eh")
print(f"Energy shift vs noiseless adj-merged: {e_noisy_adj - e_adj:.3e} Eh")

In [ ]:
# Finite-shot energy from the same ``rho_noisy`` as above: OGM layout + asymmetric readout,
# with optional REM in post-processing.
from shot_measurement import estimate_energy_from_noisy_rho_shots, sanitize_density_matrix

num_shots = int(globals()["GLOBAL_NUM_SHOTS"])
measurement_scheme = str(globals()["GLOBAL_MEASUREMENT_SCHEME"])
sampling_seed = int(globals()["GLOBAL_SAMPLING_SEED"])
epsilon = 0.1

p_0_success = np.array(globals()["GLOBAL_READOUT_P0_SUCCESS"], dtype=float)
p_1_success = np.array(globals()["GLOBAL_READOUT_P1_SUCCESS"], dtype=float)

apply_readout_noise = bool(globals()["GLOBAL_APPLY_READOUT_NOISE"])
apply_rem = True

SHADOWGROUPING_ROOT = Path(
    globals().get("SHADOWGROUPING_ROOT", "/Users/zacharyhe/shadowgrouping")
)
ogm_file = Path(
    globals().get(
        "ogm_file",
        SHADOWGROUPING_ROOT / f"haozhaowu/H4/hamil_class/ogm_outputs/OGM_H4_bond_{float(bond_length):.1f}.txt",
    )
)

print(f"OGM file: {ogm_file}  exists={ogm_file.is_file()}")

if not SHADOWGROUPING_ROOT.is_dir():
    raise FileNotFoundError(
        f"SHADOWGROUPING_ROOT does not exist: {SHADOWGROUPING_ROOT}. "
        "Set SHADOWGROUPING_ROOT to your shadowgrouping checkout."
    )
if not ogm_file.is_file():
    raise FileNotFoundError(
        f"OGM file missing: {ogm_file}. "
        "Generate/provide the H4 OGM file for this bond length or set ogm_file explicitly."
    )

rho_noisy_for_shots = sanitize_density_matrix(rho_noisy)

shot_est = estimate_energy_from_noisy_rho_shots(
    rho_noisy_for_shots,
    pauli_sum,
    list(energy_qubits),
    num_shots=num_shots,
    measurement_scheme=measurement_scheme,
    p_0_success=p_0_success,
    p_1_success=p_1_success,
    apply_rem=apply_rem,
    apply_readout_noise=apply_readout_noise,
    sampling_seed=sampling_seed,
    epsilon=epsilon,
    ogm_file=ogm_file,
    shadowgrouping_root=SHADOWGROUPING_ROOT,
)
eu = float(shot_est["energy_unmitigated"])
er = float(shot_est["energy_rem"])

print(f"Finite-shot energy (readout noise, no REM correction): {eu:.12f} Eh")
print(f"Finite-shot energy (REM readout mitigation):          {er:.12f} Eh")
print(f" (REM - reference):                            {er - e_noisy_adj:.12f} Eh")
print(f"\nReference Tr[H rho_noisy(adj-merged)] (same rho, exact DM): {e_noisy_adj:.12f} Eh")

In [ ]:
# Measurement error vs shot budget: mean(REM - reference) over independent shot seeds.
import matplotlib.pyplot as plt
import numpy as np
from shot_measurement import estimate_energy_from_noisy_rho_shots

if "rho_noisy_for_shots" not in globals():
    rho_noisy_for_shots = sanitize_density_matrix(rho_noisy)

shot_budgets = [
    8192,
    8192 * 2,
    8192 * 4,
    8192 * 10,
    8192 * 100,
]
num_replicates = 20
base_seed = int(globals().get("GLOBAL_SAMPLING_SEED", sampling_seed))

mean_errors = []
std_errors = []
for n_shots in shot_budgets:
    errs = []
    for rep in range(num_replicates):
        est = estimate_energy_from_noisy_rho_shots(
            rho_noisy_for_shots,
            pauli_sum,
            list(energy_qubits),
            num_shots=int(n_shots),
            measurement_scheme=measurement_scheme,
            p_0_success=p_0_success,
            p_1_success=p_1_success,
            apply_rem=apply_rem,
            apply_readout_noise=apply_readout_noise,
            sampling_seed=base_seed + rep,
            epsilon=epsilon,
            ogm_file=ogm_file,
            shadowgrouping_root=SHADOWGROUPING_ROOT,
        )
        errs.append(float(est["energy_rem"]) - float(e_noisy_adj))
    mean_errors.append(float(np.mean(errs)))
    std_errors.append(float(np.std(errs, ddof=1)))
    print(
        f"n_shots={n_shots:7d}: mean(REM-ref)={mean_errors[-1]:+.6e} Eh  "
        f"std={std_errors[-1]:.6e} Eh"
    )

fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=140)
ax.errorbar(
    shot_budgets,
    mean_errors,
    yerr=std_errors,
    fmt="o-",
    capsize=4,
    lw=2,
    ms=7,
    color="tab:blue",
    label=f"mean over {num_replicates} seeds",
)
ax.axhline(0.0, color="0.45", ls="--", lw=1)
ax.set_xscale("log")
ax.set_xlabel("Number of shots")
ax.set_ylabel("REM − reference (Eh)")
ax.set_title("Finite-shot measurement error vs shot budget")
ax.grid(True, alpha=0.3)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# CDR per-Pauli on H4 using symbolic decomposed template + dual near-Clifford controls.
from main_cursor_lib import (
    DEFAULT_HIGH_CZ_MULTIPLIER,
    ONE_QUBIT_GATE_DEPOL_PROB,
    TWO_QUBIT_GATE_DEPOL_PROB,
    build_cdr_parametrized_decomposed_template,
    count_clifford_nonclifford_gates,
    run_cdr_with_per_pauli_coeff_print,
)

cdr_template_circuit, cdr_qubits, cdr_symbols, cdr_template_meta = build_cdr_parametrized_decomposed_template(
    num_spatial_orbitals=int(num_spatial),
    num_layers=int(ansatz_layers),
)

if list(cdr_qubits) != list(energy_qubits):
    raise ValueError("CDR template qubits do not match energy_qubits ordering.")
if len(cdr_symbols) != len(vqe_parameters):
    raise ValueError(
        f"CDR symbol count mismatch: {len(cdr_symbols)} symbols vs {len(vqe_parameters)} parameters."
    )

target_resolver_cdr = {sym: float(val) for sym, val in zip(cdr_symbols, vqe_parameters)}

base_noise_cfg = {
    "two_qubit_depol_prob": TWO_QUBIT_GATE_DEPOL_PROB,
    "one_qubit_depol_prob": ONE_QUBIT_GATE_DEPOL_PROB,
    "high_cz_multiplier": DEFAULT_HIGH_CZ_MULTIPLIER,
}

shot_cfg = {
    "num_shots": int(globals().get("num_shots", globals()["GLOBAL_NUM_SHOTS"])),
    "measurement_scheme": str(
        globals().get("measurement_scheme", globals()["GLOBAL_MEASUREMENT_SCHEME"])
    ),
    "apply_readout_noise": bool(
        globals().get("apply_readout_noise", globals()["GLOBAL_APPLY_READOUT_NOISE"])
    ),
    "sampling_seed": int(globals().get("sampling_seed", globals()["GLOBAL_SAMPLING_SEED"])),
    "epsilon": float(globals().get("epsilon", globals()["GLOBAL_EPSILON"])),
    "ogm_file": Path(
        globals().get(
            "ogm_file",
            Path(globals().get("SHADOWGROUPING_ROOT", "/Users/zacharyhe/shadowgrouping"))
            / f"haozhaowu/H4/hamil_class/ogm_outputs/OGM_H4_bond_{float(bond_length):.1f}.txt",
        )
    ),
    "shadowgrouping_root": Path(
        globals().get("SHADOWGROUPING_ROOT", "/Users/zacharyhe/shadowgrouping")
    ),
}

readout_cal = {
    "p_0_success": np.array(
        globals().get("p_0_success", globals()["GLOBAL_READOUT_P0_SUCCESS"]),
        dtype=float,
    ),
    "p_1_success": np.array(
        globals().get("p_1_success", globals()["GLOBAL_READOUT_P1_SUCCESS"]),
        dtype=float,
    ),
}

cdr_cfg_base = {
    "num_circuits": int(globals().get("CDR_NUM_TRAINING_CIRCUITS", globals()["CDR_NUM_TRAINING_CIRCUITS"])),
    "non_clifford_params_max": int(
        globals().get("CDR_NON_CLIFFORD_PARAMS_MAX", globals().get("CDR_T_MAX_VQE", globals()["CDR_T_MAX_VQE"]))
    ),
    "target_phase_noncliff_snap_fraction": float(
        globals().get(
            "CDR_TARGET_PHASE_NONCLIFF_SNAP_FRACTION",
            globals()["CDR_TARGET_PHASE_NONCLIFF_SNAP_FRACTION"],
        )
    ),
    "seed": int(globals().get("CDR_BASE_SEED", globals()["CDR_BASE_SEED"])),
    "cdr_fit_scope": "per_pauli",
}

if not Path(shot_cfg["shadowgrouping_root"]).is_dir() or not Path(shot_cfg["ogm_file"]).is_file():
    print("Skip CDR: need valid ogm_file and SHADOWGROUPING_ROOT (same as OGM shot cell).")
else:
    mit_pp = run_cdr_with_per_pauli_coeff_print(
        ansatz_circuit=cdr_template_circuit,
        observable_h=pauli_sum,
        qubits=list(cdr_qubits),
        target_resolver=target_resolver_cdr,
        target_params=target_resolver_cdr,
        symbols=cdr_symbols,
        base_noise_cfg=base_noise_cfg,
        shot_cfg=shot_cfg,
        readout_cal=readout_cal,
        cdr_cfg=cdr_cfg_base,
        simulator_seed=int(globals().get("random_seed", globals()["GLOBAL_RANDOM_SEED"])),
    )

    print("\nCDR (per-pauli)")
    print(
        f"raw finite-shot (unmit / REM): {float(mit_pp['unmit_target']):.12f} / {float(mit_pp['rem_target']):.12f} Eh"
    )
    print(
        "cdr corrected (unmit / REM): "
        f"{float(mit_pp['cdr_unmit_corrected']):.12f} / {float(mit_pp['cdr_rem_corrected']):.12f} Eh"
    )
    print(f"reference exact noiseless (adj-merged): {float(e_adj):.12f} Eh")
    print(f"Energy error (reference - cdr_rem): {float(e_adj) - float(mit_pp['cdr_rem_corrected']):.12f} Eh")

    training_constraints = mit_pp.get("cdr_training_constraints", {})
    training_summary = mit_pp.get("cdr_training_summary", {})
    if training_constraints:
        print("\nCDR training constraints:")
        print(training_constraints)
    if training_summary:
        print("CDR training achieved summary:")
        print(training_summary)

    gate_counts = count_clifford_nonclifford_gates(cdr_template_circuit, target_resolver_cdr)
    
    print("\nFinal circuit details (CDR target circuit):")
    print(f"total gates: {int(gate_counts['total_gates'])}")
    print(f"non-Clifford gates: {int(gate_counts['non_clifford_gates'])}")
    print(f"Clifford gates: {int(gate_counts['clifford_gates'])}")
    print(f"symbol coupling metadata entries: {len(cdr_template_meta.get('symbol_metadata', {}))}")

In [ ]:
3*(3*num_spatial-2)*6+4 # is the number of cliffords in the decomposed template

In [ ]:
# Cumulative histograms for H4 per-Pauli CDR fit coefficients (REM branch):
# exact_k ~= a_k * noisy_k + b_k
import numpy as np
import matplotlib.pyplot as plt

if "mit_pp" not in globals():
    raise RuntimeError("Run the H4 CDR cell above first to create `mit_pp`.")

models = dict(mit_pp.get("cdr_models", {}))
coeffs_rem = np.asarray(models.get("coeffs_rem_to_exact_per_term", []), dtype=float)
r2_vals = np.asarray(models.get("r2_rem_to_exact_per_term", []), dtype=float)

if coeffs_rem.size == 0:
    raise RuntimeError("No per-Pauli coefficients found in mit_pp['cdr_models'].")
if coeffs_rem.ndim != 2 or coeffs_rem.shape[1] < 2:
    raise RuntimeError(f"Unexpected coeff shape: {coeffs_rem.shape}")
if r2_vals.size == 0:
    raise RuntimeError(
        "No per-Pauli R^2 values found in mit_pp['cdr_models']."
    )
if r2_vals.shape[0] != coeffs_rem.shape[0]:
    raise RuntimeError(
        f"R^2 length {r2_vals.shape[0]} != coefficient rows {coeffs_rem.shape[0]}"
    )

a_vals = np.asarray(coeffs_rem[:, 0], dtype=float)
b_vals = np.asarray(coeffs_rem[:, 1], dtype=float)

# Panel a only: exclude terms where both slope and intercept are ~0.
zero_pair_mask = np.isclose(a_vals, 0.0, atol=5e-3) & np.isclose(b_vals, 0.0, atol=5e-3)
a_vals_panel_a = a_vals[~zero_pair_mask]
if a_vals_panel_a.size == 0:
    raise RuntimeError("Panel a: all terms have both slope and intercept near zero (atol=5e-3).")

def _empirical_cdf(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    x = np.sort(np.asarray(values, dtype=float))
    y = np.arange(1, len(x) + 1, dtype=float) / float(len(x))
    return x, y

x_a, y_a = _empirical_cdf(a_vals_panel_a)
x_b, y_b = _empirical_cdf(b_vals)
x_c, y_c = _empirical_cdf(r2_vals)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.8), dpi=140)
h4_color = "tab:blue"

axes[0].plot(x_a, y_a, color=h4_color, lw=2.2, label="H4")
axes[0].set_xlabel("Fit slope")
axes[0].set_ylabel("Integrated histogram")
axes[0].set_xlim(0, 5.2)
axes[0].set_ylim(0.0, 1.0)
axes[0].text(-0.16, 1.02, "a", transform=axes[0].transAxes, fontsize=20, fontweight="bold")

axes[1].plot(x_b, y_b, color=h4_color, lw=2.2, label="H4")
axes[1].set_xlabel("Fit intercept")
axes[1].set_xlim(-0.14, 0.14)
axes[1].set_ylim(0.0, 1.0)
axes[1].text(-0.16, 1.02, "b", transform=axes[1].transAxes, fontsize=20, fontweight="bold")

axes[2].plot(x_c, y_c, color=h4_color, lw=2.2, label="H4")
axes[2].set_xlabel(r"Goodness of fit ($R^2$)")
axes[2].set_xlim(0.0, 1.0)
axes[2].set_ylim(0.0, 1.0)
axes[2].text(-0.16, 1.02, "c", transform=axes[2].transAxes, fontsize=20, fontweight="bold")
axes[2].legend(loc="lower right", frameon=True)

for ax in axes:
    ax.grid(alpha=0.18)

plt.tight_layout()
plt.show()

print(f"H4 terms total: {len(a_vals)}")
print(f"Panel a terms (excl. a≈b≈0): {len(a_vals_panel_a)} (dropped {int(zero_pair_mask.sum())})")
print(f"Panel a slope (a): mean={np.mean(a_vals_panel_a):.6f}, std={np.std(a_vals_panel_a):.6f}")
print(f"All terms slope (a): mean={np.mean(a_vals):.6f}, std={np.std(a_vals):.6f}")
print(f"Intercept (b): mean={np.mean(b_vals):.6f}, std={np.std(b_vals):.6f}")
print(f"R^2: mean={np.mean(r2_vals):.6f}, std={np.std(r2_vals):.6f}")